In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from google.cloud import storage
from collections import Counter

import pandas as pd
import numpy as np
import string
import time
import copy
import random
import json
import requests
import os
import hashlib
import warnings


import preprocess
import model_arch
import update_model

In [2]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

DEVICE = get_device()
print('Using device:', str(DEVICE).upper(), "\n")

MODEL_DIR="./model"
DATA_PATH = "comb.txt"

TRAIN_DATA_BUCKET = "cbow-training-data-1f656"
MODEL_DATA_BUCKET = "cbow-model-data-1f656"

MPS is available
Using device: MPS 



In [ ]:
# === Run this code for the first model initialization ---- 

"""
EMBEDDING_SIZE = 256
WINDOW_SIZE = 2
BATCH_SIZE = 128

## word2idx is a vocabulary (vocab) that has the format {"word_1: idx_1, "word_2", idx_2, ...}
word2idx, idx2word = preprocess.get_index(f"data/{DATA_PATH}")

sequence, vocab_size = preprocess.get_sequence(DATA_PATH)
dataset = model_arch.CBOWDataset(sequence, WINDOW_SIZE)
model = model_arch.CBOWModel(vocab_size, EMBEDDING_SIZE)

## Save model configuration in the JSON file

hash_5 = ''.join(random.choices(string.ascii_letters + string.digits, k=5))
model_params = {
    "embedding_shape": tuple(model.embedding.weight.data.shape),
    "window_size": WINDOW_SIZE,
    "batch_size": BATCH_SIZE,
    "model_path": f"model/{MODEL_NAME}_{hash_5}.pt",
    "word2idx": word2idx,
}
with open(f"model/{MODEL_NAME}.json", "w") as f:
    json.dump(model_params, f, indent=4)
"""

In [ ]:
# Download training data from the GCS bucket
preprocess.download_data_from_gcs(
    TRAIN_DATA_BUCKET,
    DATA_PATH
)

# Download pretrained model and its configuration
preprocess.download_model_from_gcs(
    bucket_name=MODEL_DATA_BUCKET,
    download_dir=MODEL_DIR,
)

In [4]:
model, cfg = update_model.initiate_model(f"data/{DATA_PATH}")

# Define model params
embedding_size = cfg['embedding_size']
window_size    = cfg['window_size']
batch_size     = cfg['batch_size']
word2idx       = cfg['word2idx']

 CBOW Incremental Update
   new data : data/comb.txt
   device   : MPS
 -- Loaded weights from ./model/model_cpu.pt
 -- Skipping 'data/comb.txt': already ingested (hash match)
 -- Vocab unchanged; no layer resizing needed

 -- Done ✓


In [ ]:
# === MODEL TRAINING
model = model.to(DEVICE)

epochs = 20
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    start = time.perf_counter()
    
    total_loss = 0
    for context, target in dataloader:
        context, target = context.to(DEVICE), target.to(DEVICE) # move to GPU
        
        optimizer.zero_grad()
        logits = model(context)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    end = time.perf_counter()

    print(f" ---- Epoch {epoch+1}, Loss: {total_loss:.4f}, Execution time: {end - start:.6f}")

In [5]:
# # === Save model locally
# update_model.save(
#     model,
#     cfg,
#     model_dir="model",
#     config_path="model/model_config.json"
# )

# === Save in GCS bucket
client = storage.Client() # .from_service_account_info(json_key)
bucket = client.bucket(MODEL_DATA_BUCKET)

blobs = client.list_blobs(MODEL_DATA_BUCKET, delimiter='/')
_ = list(blobs)

latest_version = max([int(v.split('-')[1].replace('/', '')) for v in blobs.prefixes])
print(latest_version)

/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/dpakki/tempNotes/Image_Processing/cbow_pytorch/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


1


In [6]:
model = model.to(DEVICE)
word_embeddings = model.embedding.weight.detach()

# Example prediction
sentense = "i want to break free".lower()
context_words = []

split_sentense = sentense.split(" ")

if len(split_sentense) != window_size * 2 + 1:
    print("ERROR: sentense does not have the сorrect lenght for testinng")
else:
    for w in split_sentense:
        context_words.append(w)
    del context_words[window_size]

print("Context words: ", context_words)

Context words:  ['i', 'want', 'break', 'free']


In [7]:
top_k = 5
model = model.to(DEVICE)
model.eval()

idx2word = {}
for k, v in word2idx.items():
    idx2word[v] = k

context_idxs = torch.tensor(
    [[word2idx[w] for w in context_words]]
)
context_idxs = context_idxs.to(DEVICE)

with torch.no_grad():
    logits = model(context_idxs)
    probs = torch.softmax(logits, dim=1)

top_idxs = torch.topk(probs, top_k).indices[0]
top_idxs
result = [idx2word[idx.item()] for idx in top_idxs]
print(result)

['don’t', 'will', 'to', 'food', 'for']


In [14]:
word_1 = "white"
word_2 = "green"

v1 = word_embeddings[word2idx[word_1]]
v2 = word_embeddings[word2idx[word_2]]
print(f"Original: {F.cosine_similarity(v1, v2, dim=0).item()}")

Original: 0.4399053752422333


In [9]:
# coincides = 0
# w1_pos = 0
# w2_pos = 0
# for i in range(len(word_emb1)):
#     if word_emb1[i] == 1.0:
#         w1_pos += 1
#     if word_emb2[i] == 1.0:
#         w2_pos += 1
#     if word_emb1[i] == 1.0 and word_emb2[i] == 1.0:
#         coincides += 1

# print(f"Number of 1-position for {word_1}: {w1_pos}")
# print(f"Number of 1-position for {word_2}: {w2_pos}")
# print(f"Number of coincides: {coincides}")

In [10]:
words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

In [12]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

# thresholds = [0.1,0.2,0.3,0.4]
thresholds = [i for i in range(1, 11)]

for t in thresholds:
    # filtering
    B = (embeddings >= t).int()
    # C = torch.cov(B.T)
    C = B @ B.T

    print(f" -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_1-cov_matrix_{t}.csv")

 -- threshold = 1, nonzoer(B) = 1088
 -- threshold = 2, nonzoer(B) = 774
 -- threshold = 3, nonzoer(B) = 568
 -- threshold = 4, nonzoer(B) = 408
 -- threshold = 5, nonzoer(B) = 296
 -- threshold = 6, nonzoer(B) = 220
 -- threshold = 7, nonzoer(B) = 162
 -- threshold = 8, nonzoer(B) = 115
 -- threshold = 9, nonzoer(B) = 78
 -- threshold = 10, nonzoer(B) = 59


In [13]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

deltas = [1, 2, 3] # [0.02,0.05]

for d in deltas:
    # filtering
    B = (torch.abs(embeddings) <= d).int()
    C = B @ B.T

    print(f"-- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/filtration_2-cov_matrix_{d}.csv")

-- delta = 1, nonzoer(B) = 796
-- delta = 2, nonzoer(B) = 1540
-- delta = 3, nonzoer(B) = 2103
